In [1]:
import os
import numpy as np
import nibabel as nib
import h5py
import gc

In [ ]:
# downloads a specific beta-series NIfTI file

def download_nsd_data(subject, session, save_path="/home/jovyan/cache/memoryNSD/"):
    subj_str = f"subj{subject:02d}"
    session_str = f"session{session:02d}"

    url = f"https://natural-scenes-dataset.s3.amazonaws.com/nsddata_betas/ppdata/{subj_str}/func1mm/betas_fithrf_GLMdenoise_RR/betas_{session_str}.nii.gz"
    file_path = os.path.join(save_path, f"{subj_str}_betas_{session_str}.nii.gz")

    os.system(f"wget -q -O {file_path} {url}")
    print(f"Downloaded beta file: {file_path}")
    return file_path


In [32]:
# for loading fMRI data

def load_betas(file_path):
    img = nib.load(file_path)
    return np.array(img.dataobj) # img.get_fdata(dtype=np.float32) / 300.0; divide by 300 later

def load_mask(file_path):
    img = nib.load(file_path)
    return img.get_fdata(dtype=np.float32)



In [6]:
# This function downloads and saves the MTL mask for a specific NSD subject to a local directory for later use

def download_mtl_mask(subject, save_path="/home/jovyan/cache/memoryNSD/"):
    subj_str = f"subj{subject:02d}"
    url = f"https://natural-scenes-dataset.s3.amazonaws.com/nsddata/ppdata/{subj_str}/func1mm/roi/MTL.nii.gz"
    file_path = os.path.join(save_path, f"{subj_str}_MTL.nii.gz")

    os.system(f"wget -q -O {file_path} {url}")
    print(f"Downloaded MTL mask: {file_path}")
    return file_path

In [7]:
# This function, extract_roi_betas, extracts and normalizes beta values from a specified region of interest (ROI) in fMRI data

def extract_roi_betas(fmri_data, roi_indices):
    voxel_betas = fmri_data[roi_indices]  # shape: voxels x trials
    voxel_betas = voxel_betas.T           # trials x voxels
    voxel_betas = (voxel_betas - voxel_betas.mean(axis=0)) / (voxel_betas.std(axis=0) + 1e-6)
    return voxel_betas

# did a z-score within each session to ensure when we compare across sessions, they are all standardised

In [8]:
# This function, extract_mtl_rois, extracts and processes beta values from 10 subregions of the medial temporal lobe (MTL) using a labeled ROI mask.

def extract_mtl_rois(fmri_data, mtl_data):
    mtl_rois = {}
    for roi_label in range(1, 11):  # Labels 1 to 10
        roi_indices = np.where(mtl_data == roi_label)
        if len(roi_indices[0]) > 0:
            roi_betas = extract_roi_betas(fmri_data, roi_indices)
            mtl_rois[roi_label] = roi_betas / 300.00
    return mtl_rois


In [9]:
# This function saves a nested dictionary of data into an HDF5 file, organizing it by session and ROI.
def save_hdf5(data, file_path):
    with h5py.File(file_path, 'w') as hf:
        for session, rois in data.items():
            session_group = hf.create_group(f"session{session:02d}")
            for roi_label, roi_data in rois.items():
                session_group.create_dataset(f"roi_{roi_label}", data=roi_data)
    print(f"Saved all_mtl_sessions to {file_path}")

In [ ]:
# Create directory if needed, Loop over subjects (1 to 8), Download MTL mask, Loop over sessions (1 to 40), Save all sessions' MTL data, then The downloaded MTL mask file is removed from disk to save space


os.makedirs("/home/jovyan/cache/memoryNSD/", exist_ok=True)


for subject in range(1, 9):
    print(f"\n--- Processing Subject {subject:02d} ---")
    all_mtl_sessions = {}

    mtl_mask_path = download_mtl_mask(subject)
    MTL_data = load_mask(mtl_mask_path)

    for session in range(1, 41):
        print(f"Processing Session {session} for Subject {subject:02d}...")
        beta_path = download_nsd_data(subject=subject, session=session)

        try:
            betas = load_betas(beta_path)
            all_mtl_sessions[session] = extract_mtl_rois(betas, MTL_data)
            del betas
            gc.collect()
            os.remove(beta_path)
        except Exception as e:
            print(f"Error processing subj{subject:02d} session{session:02d}: {e}")

    save_mtl_path = f"/home/jovyan/cache/memoryNSD/subj{subject:02d}_all_mtl_sessions.h5"
    save_hdf5(all_mtl_sessions, save_mtl_path)
    os.remove(mtl_mask_path)



--- Processing Subject 01 ---
Downloaded MTL mask: /home/jovyan/cache/memoryNSD/subj01_MTL.nii.gz
Processing Session 1 for Subject 01...


In [ ]:
import gc
gc.collect()





In [ ]:

betas = load_betas("/home/jovyan/cache/memoryNSD/subj01_betas_session01.nii.gz")


In [12]:
MTL_data = load_mask("/home/jovyan/cache/memoryNSD/subj01_MTL.nii.gz")
np.unique(MTL_data)

array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
      dtype=float32)

In [27]:
import nibabel as nib

betas = nib.load("/home/jovyan/cache/memoryNSD/subj01_betas_session01.nii.gz")
betas = np.array(betas.dataobj)

In [28]:
betas.shape

(145, 186, 148, 750)

In [15]:
mask = (MTL_data == 1)

In [29]:
temp = betas[mask,:]

# the code is accidentally making too many copies of the array

In [31]:
temp.shape

(2163, 750)